# 4. Model Comparison and Selection

This notebook compares multiple machine learning algorithms to select the best-performing model for credit default prediction.

The main objectives are:

- Load the preprocessed training and testing datasets
- Define and train multiple candidate models:
  - Baseline Logistic Regression
  - Random Forest Classifier
  - XGBoost Classifier
  - LightGBM Classifier
- Evaluate candidate models across key classification and probability metrics:
  - ROC-AUC and PR-AUC
  - Recall, Precision, and F1-Score
  - Brier Score and Log Loss
- Compare models using cross-validation and test set evaluation
- Select the best overall model for probability calibration and downstream risk scoring

A multi-metric evaluation strategy is used to balance raw predictive power, class imbalance handling (default detection), and reliable probability estimation.

In [2]:
import xgboost

print(xgboost.__version__)

3.4.1


In [3]:
# Loading processed data set

import pandas as pd
import numpy as np

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (24000, 39)
X_test: (6000, 39)
y_train: (24000,)
y_test: (6000,)


In [4]:
# Define the features

categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("Numerical features:", len(numerical_features))

Categorical features: ['SEX', 'EDUCATION', 'MARRIAGE']
Numerical features: 36


In [5]:
# Create preprocessing pipelines

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [6]:
# Logistic Regression preprocessing

numeric_pipeline_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer([
    ("num", numeric_pipeline_lr, numerical_features),
    ("cat", categorical_pipeline_lr, categorical_features)
])

### Random Forest

In [8]:
from sklearn.ensemble import RandomForestClassifier

In [9]:
rf_preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numerical_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

In [10]:
rf_model = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

Train Random Forest

In [17]:
print("Training Random Forest...")

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

Training Random Forest...
Random Forest training completed.


In [18]:
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

### XGBoost

In [11]:
from xgboost import XGBClassifier

In [13]:
xgb_preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numerical_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

In [14]:
xgb_model = Pipeline([
    ("preprocessor", xgb_preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

Train XGBoost

In [15]:
print("Training XGBoost...")

xgb_model.fit(X_train, y_train)

print("XGBoost training completed.")

Training XGBoost...
XGBoost training completed.


In [16]:
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

Create a comparison function

In [19]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)

def evaluate_model(name, y_true, y_pred, y_proba):

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "PR-AUC": average_precision_score(y_true, y_proba),
        "Brier Score": brier_score_loss(y_true, y_proba),
        "Log Loss": log_loss(y_true, y_proba)
    }

### Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print("Training Logistic Regression...")

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

print("Logistic Regression training completed.")

Training Logistic Regression...
Logistic Regression training completed.


Compare all models

In [21]:
results = []

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        lr_pred,
        lr_proba
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred,
        rf_proba
    )
)

results.append(
    evaluate_model(
        "XGBoost",
        y_test,
        xgb_pred,
        xgb_proba
    )
)

results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Brier Score,Log Loss
0,Logistic Regression,0.808167,0.625399,0.643322,0.297664,0.407007,0.743170,0.503094,0.142842,0.452188
1,Random Forest,0.810333,0.651881,0.620076,0.367747,0.461684,0.760205,0.536040,0.139768,0.445196
2,XGBoost,0.816167,0.654547,0.650538,0.364732,0.467407,0.779569,0.554921,0.135553,0.431365
